# 04 - Performance Analytics Engine
This notebook outlines the calculation of key risk and return metrics for mutual funds:
1. **Daily Returns**: $R_t = \frac{NAV_t}{NAV_{t-1}} - 1$
2. **CAGR**: $CAGR = \left(\frac{\text{Ending NAV}}{\text{Beginning NAV}}\right)^{\frac{252}{n}} - 1$ (using trading days logic)
3. **Sharpe Ratio**: $\text{Sharpe} = \frac{R_p - R_f}{\sigma_p}$
4. **Sortino Ratio**: $\text{Sortino} = \frac{R_p - R_f}{\sigma_{down}}$ (downside deviation only)
5. **Beta**: $\beta = \frac{\text{Cov}(R_p, R_m)}{\text{Var}(R_m)}$
6. **Alpha**: $\alpha = R_p - [R_f + \beta(R_m - R_f)]$
7. **Max Drawdown**: Peak to trough maximum fall


In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import pathlib

base_dir = pathlib.Path('D:/New folder/bluestock_mf_capstone')
engine = create_engine(f'sqlite:///{base_dir}/data/db/bluestock_mf.db')


## Trigger Performance Computations
We can load the calculated metrics from `fact_performance` in the database, which are updated via `compute_metrics.py`.


In [2]:
df_perf = pd.read_sql('''
    SELECT f.fund_name, f.category, p.*
    FROM fact_performance p
    JOIN dim_fund f ON p.amfi_code = f.amfi_code
''', engine)
display(df_perf.sort_values(by='cagr_3y', ascending=False).head(10))


,fund_name,category,performance_id,amfi_code,risk_level,benchmark_name,cagr_1y,cagr_3y,cagr_5y,sharpe_ratio,sortino_ratio,beta,alpha,max_drawdown,morningstar_rating,risk_grade
19,SBI Bluechip Fund - Regular Plan - Growth,Equity Large Cap,20,119551,Moderate,NIFTY100,0.3265,0.2829,0.1977,1.8750,1.6628,-0.0313,0.1655,0.1501,4,Moderate
38,DSP Midcap Fund - Regular - Growth,Equity Mid Cap,39,149323,High,NIFTY_MIDCAP150,0.1304,0.2533,0.2287,1.2549,1.0788,-0.0020,0.1897,0.1725,4,High
34,Mirae Asset Large Cap Fund - Regular - Growth,Equity Large Cap,35,148567,Moderate,NIFTY100,0.0441,0.2503,0.1803,1.5430,1.3333,0.0243,0.1924,0.1127,5,Moderate
2,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,Equity Mid Cap,3,100033,High,NIFTY_MIDCAP150,0.3533,0.2324,0.2266,1.0457,0.9187,0.0057,0.1940,0.1622,5,High
3,ABSL Frontline Equity Fund - Regular - Growth,Equity Large Cap,4,101206,Moderate,NIFTY100,0.3732,0.2208,0.1761,1.2649,1.1284,0.0215,0.1526,0.1129,5,Moderate
30,Kotak Flexicap Fund - Regular - Growth,Equity Flexi Cap,31,120843,Moderately High,NIFTY500,0.2374,0.2161,0.2287,1.1243,1.0224,-0.0223,0.1949,0.1297,5,Moderately High
16,Axis Midcap Fund - Regular - Growth,Equity Mid Cap,17,119094,High,NIFTY_MIDCAP150,0.2125,0.2099,0.2347,0.8835,0.7945,-0.0657,0.1860,0.2096,5,High
36,Mirae Asset Tax Saver Fund - Regular - Growth,Equity ELSS,37,148569,High,NIFTY500,0.3945,0.2091,0.1887,0.9642,0.8695,0.0187,0.2016,0.1640,5,High
21,SBI Small Cap Fund - Regular Plan - Growth,Equity Small Cap,22,119598,Very High,BSE_SMALLCAP,0.4027,0.1923,0.1850,0.5993,0.5310,-0.0226,0.2164,0.2871,5,Very High
39,DSP Small Cap Fund - Regular - Growth,Equity Small Cap,40,149324,Very High,BSE_SMALLCAP,0.3345,0.1745,0.2190,0.5215,0.4589,0.0121,0.2144,0.3117,4,Very High


## Code Demonstration: Rolling NAV Averages (SQL Window Function)
We query the 30-day moving average of fund `119551` using SQL Window functions.


In [3]:
query = '''
SELECT 
    date_id,
    nav,
    AVG(nav) OVER (
        ORDER BY date_id 
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) as rolling_30d_avg
FROM fact_nav
WHERE amfi_code = 119551
LIMIT 15;
'''
with engine.connect() as conn:
    df_rolling = pd.read_sql(text(query), conn)
display(df_rolling)


,date_id,nav,rolling_30d_avg
0,2022-01-01,54.3856,54.385600
1,2022-01-02,54.3856,54.385600
2,2022-01-03,54.3856,54.385600
3,2022-01-04,54.3474,54.376050
4,2022-01-05,54.6869,54.438220
5,2022-01-06,55.4550,54.607683
6,2022-01-07,55.3692,54.716471
7,2022-01-08,55.3692,54.798063
8,2022-01-09,55.3692,54.861522
9,2022-01-10,55.2835,54.903720
